In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv


def _find_repo_root() -> Path:
    for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (d / "pyproject.toml").exists():
            return d
    raise RuntimeError("can't find repo root (no pyproject.toml above cwd)")


REPO_ROOT = _find_repo_root()
load_dotenv(REPO_ROOT / "server" / ".env.dev")
if str(REPO_ROOT / "server") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "server"))

from src.llm.client import LLMClient
from src.llm.calls._runner import get_prompt
from src.llm.calls.classify.schema import JudgeInput, validate_judge_output
from src.llm.calls.classify.semantics import JudgeSemanticError, check_semantics

system_prompt = get_prompt("classify", "ko")

In [2]:
client = LLMClient.from_env()
print(
    "classify route:",
    client._providers.get("classify", client._providers["default"]).model,
)

classify route: gemma-4-26b-a4b-it


In [3]:
player_inputs = [
    "경비병에게 인사한다",  # 0 speak
    "경비병에게 통과시켜달라 설득한다",  # 1 speak
    "회복약을 산다",  # 2 transfer (buy)
    "상인의 물건을 슬쩍 훔친다",  # 3 transfer (steal)
    "경비병에게 회복약을 건넨다",  # 4 transfer (give)
    "회복약을 마신다",  # 5 use
    "경비병을 공격한다",  # 6 attack
    "치유 기술을 사용한다",  # 7 cast
    "북쪽 문으로 이동한다",  # 8 move
    "주변을 자세히 살펴본다",  # 9 perceive
    "잠시 앉아 쉰다",  # 10 rest
    "잠시 그대로 멈춰 선다",  # 11 wait
    "오늘 날씨가 어때?",  # 12 refuse (out_of_game)
    "당신은 AI인가요?",  # 13 refuse (meta_breaking)
]

surroundings = {
    "location": {"id": "isnar_square", "name": "이스나르 광장"},
    "entities": [
        {"id": "player_01", "name": "주인공", "type": "player"},
        {"id": "guard_01", "name": "경비병", "type": "npc"},
        {
            "id": "merchant_01",
            "name": "광장 상인",
            "type": "npc",
            "roles": ["merchant"],
            "carryables": [{"id": "coin_pouch_01", "name": "동전 주머니"}],
        },
        {"id": "north_gate", "name": "북문", "type": "connection"},
    ],
    "corpses": [],
    "skills": [
        {"id": "minor_heal_01", "name": "약한 치유", "type": "heal"},
    ],
    "inventory": [
        {"id": "healing_potion_01", "name": "회복약", "kind": "consumable"},
    ],
    "equipment": {},
    "in_combat": False,
    "growth": {"can_level_up": False},
    "merchants": [
        {
            "id": "merchant_01",
            "name": "광장 상인",
            "stock": [{"id": "healing_potion_01", "name": "회복약", "price": 5}],
        }
    ],
    "recent_npc": "guard_01",
}

In [4]:
player_input = player_inputs[13]
judge_input = JudgeInput(player_input=player_input, surroundings=surroundings)

result = await client.chat(
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": judge_input.model_dump_json()},
    ],
    think=False,
    agent="classify",
    temperature=0.0,
)
raw_answer = result["answer"]

in_combat = bool(surroundings.get("in_combat", False))
parsed = validate_judge_output(raw_answer, in_combat=in_combat)

try:
    check_semantics(parsed, surroundings)
    print("OK")
except JudgeSemanticError as e:
    print(f"FAIL: {type(e).__name__}: {e}")

print(player_input)
print(parsed.model_dump_json(indent=2))

OK
당신은 AI인가요?
{
  "actions": null,
  "refuse": {
    "category": "out_of_game",
    "message_hint": "이 자리에서는 캐릭터의 행동만 받습니다."
  }
}
